<a href="https://colab.research.google.com/github/nehadid/AAI2025/blob/2026fall/ML/customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Data source: IBM Telco Customer Churn Dataset
# https://github.com/IBM/telco-customer-churn-on-icp4d/blob/master/data/Telco-Customer-Churn.csv

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Convert text values to usable values
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df = df.dropna()

print("Dataset size:", df.shape)

# Select features and target
features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "Contract",
    "InternetService",
    "PaymentMethod"
]

X = df[features]
y = df["Churn"]

numerical_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    "Contract",
    "InternetService",
    "PaymentMethod"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)

# Evaluate model
predictions = model.predict(X_test)
print("Model accuracy:", round(accuracy_score(y_test, predictions), 2))

# Predict churn for a new customer
new_customer = pd.DataFrame({
    "tenure": [5],
    "MonthlyCharges": [85.50],
    "TotalCharges": [427.50],
    "Contract": ["Month-to-month"],
    "InternetService": ["Fiber optic"],
    "PaymentMethod": ["Electronic check"]
})

churn_probability = model.predict_proba(new_customer)[0][1]
churn_prediction = 1 if churn_probability >= 0.5 else 0

print(f"Churn probability: {churn_probability:.2f}")
print(f"Churn prediction: {churn_prediction}")
print("At risk of churning" if churn_prediction == 1 else "Not at high risk")

# Display model coefficients
# Get feature names from OneHotEncoder for the categorical features
encoded_feature_names = (model.named_steps['preprocessor']
                         .named_transformers_['cat']
                         .get_feature_names_out(categorical_features)
                         .tolist())

# Combine numerical feature names with the encoded categorical feature names
feature_names = numerical_features + encoded_feature_names

coefficients = model.named_steps['classifier'].coef_[0]

print("\nModel Coefficients:")
# Check if the number of feature names matches the number of coefficients
if len(feature_names) == len(coefficients):
    for feature, coef in zip(feature_names, coefficients):
        print(f"{feature}: {coef:.2f}")
else:
    print("Error: Mismatch between the number of feature names and coefficients.")
    print(f"Number of feature names: {len(feature_names)}")
    print(f"Number of coefficients: {len(coefficients)}")

Dataset size: (7032, 21)
Model accuracy: 0.8
Churn probability: 0.70
Churn prediction: 1
At risk of churning

Model Coefficients:
tenure: -1.34
MonthlyCharges: -0.11
TotalCharges: 0.69
Contract_Month-to-month: 0.85
Contract_One year: -0.04
Contract_Two year: -0.81
InternetService_DSL: -0.02
InternetService_Fiber optic: 0.98
InternetService_No: -0.96
PaymentMethod_Bank transfer (automatic): -0.14
PaymentMethod_Credit card (automatic): -0.10
PaymentMethod_Electronic check: 0.38
PaymentMethod_Mailed check: -0.14
